In [1]:
import pandas as pd
import re

# ============================================================
# #(1) 表示設定（省略「...」を出さない）
# ============================================================
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 0)
pd.set_option("display.max_colwidth", None)

# ============================================================
# #(2) 共通：code正規化（文字列のまま）
#   - 前後空白除去
#   - 全角数字→半角
# ============================================================
def normalize_code_str(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .str.strip()
         .str.translate(str.maketrans("０１２３４５６７８９", "0123456789"))
    )

# ============================================================
# #(3) 02_OWNlist × 01_IDmap を統合して merged_df を作る
#   - 「証券会社」列も出力（OWNlistに broker or 証券会社 がある前提）
# ============================================================
own_df = pd.read_csv("Data/02_OWNlist.csv", dtype={"code": "string"})
idmap_df = pd.read_csv("Data/01_IDmap.csv", dtype={"code": "string"})

own_df["code"] = normalize_code_str(own_df["code"])
idmap_df["code"] = normalize_code_str(idmap_df["code"])

# --- 証券会社列の自動判定（broker優先、無ければ「証券会社」） ---
if "broker" in own_df.columns:
    broker_col = "broker"
elif "証券会社" in own_df.columns:
    broker_col = "証券会社"
else:
    raise KeyError("02_OWNlist.csv に 'broker' または '証券会社' 列が見つかりません。列名を確認してください。")

own_cols = ["code", "所有者", broker_col, "口座区分", "株数", "取得単価"]
idmap_cols = ["code", "name", "industry_33"]

own_df = own_df[own_cols].rename(columns={broker_col: "証券会社"})
idmap_df = idmap_df[idmap_cols]

merged_df = pd.merge(
    own_df,
    idmap_df,
    on="code",
    how="left"
)

# ============================================================
# #(4) 03_haitou.csv から「採用する年間配当」を作る
#   ルール：
#   - 予想（"予"付き）があれば、その中で最新年度を採用
#   - 予想が無ければ、実績（"予"なし）の最新年度を採用
#   - 両方無ければ NaN
# ============================================================
# BOM対策で utf-8-sig を推奨（"﻿code"問題の予防）
haitou_df = pd.read_csv("Data/03_haitou.csv", dtype={"code": "string"}, encoding="utf-8-sig")

# もし列名が "﻿code" になっていた場合も救済
if "code" not in haitou_df.columns:
    for c in haitou_df.columns:
        if str(c).replace("\ufeff", "") == "code":
            haitou_df = haitou_df.rename(columns={c: "code"})
            break

haitou_df["code"] = normalize_code_str(haitou_df["code"])

year_cols = [c for c in haitou_df.columns if re.fullmatch(r"\d{4}", str(c))]
year_cols_sorted = sorted(year_cols, key=lambda x: int(x))

def parse_div_cell(x):
    if pd.isna(x):
        return (False, None)
    s = str(x).strip()
    if s == "":
        return (False, None)
    is_forecast = "予" in s
    m = re.search(r"(\d+(?:\.\d+)?)", s)  # 小数対応
    if not m:
        return (is_forecast, None)
    return (is_forecast, float(m.group(1)))

def pick_latest_dividend(row):
    latest_forecast = (None, None)  # (year, value)
    latest_actual = (None, None)

    for y in year_cols_sorted:
        is_fc, val = parse_div_cell(row.get(y))
        if val is None:
            continue
        year_i = int(y)
        if is_fc:
            if latest_forecast[0] is None or year_i > latest_forecast[0]:
                latest_forecast = (year_i, val)
        else:
            if latest_actual[0] is None or year_i > latest_actual[0]:
                latest_actual = (year_i, val)

    if latest_forecast[0] is not None:
        return pd.Series({"年間配当_採用": latest_forecast[1], "年間配当_年度": latest_forecast[0], "年間配当_区分": "予想"})
    if latest_actual[0] is not None:
        return pd.Series({"年間配当_採用": latest_actual[1], "年間配当_年度": latest_actual[0], "年間配当_区分": "実績"})
    return pd.Series({"年間配当_採用": pd.NA, "年間配当_年度": pd.NA, "年間配当_区分": pd.NA})

picked = haitou_df.apply(pick_latest_dividend, axis=1)

haitou_pick_df = pd.concat([haitou_df[["code"]].copy(), picked], axis=1)
haitou_pick_df = (
    haitou_pick_df
    .sort_values(["code", "年間配当_年度"])
    .drop_duplicates(subset=["code"], keep="last")
)

# ============================================================
# #(5) merged_df に配当列を追加
# ============================================================
merged_df = pd.merge(
    merged_df,
    haitou_pick_df,
    on="code",
    how="left"
)

merged_df = merged_df.rename(columns={"年間配当_採用": "年間配当"})

# ============================================================
# #(6) 表示（統合一覧）
# ============================================================
display(merged_df)

# （任意）配当が取れなかった銘柄チェック
no_div = merged_df[merged_df["年間配当"].isna()][["code", "name", "industry_33"]].drop_duplicates()
print("配当取得できない銘柄数:", len(no_div))
display(no_div)


import datetime as _dt

# ============================================================
# #(7) 集計対象銘柄を条件付きで抽出する関数
#   条件：
#   - 年間配当があり、かつ
#     年間配当_年度 が「今年 or 昨年」
#   - (所有者, 証券会社, 口座区分) の組が指定リストに一致
#
# owners_brokers_accounts :
#   [
#     ("Y", "SBI", "NISA"),
#     ("T", "楽天", "特定"),
#     ...
#   ]
#   ※ None または "*" を指定するとワイルドカード
# ============================================================
def filter_target_stocks(
    merged_df: pd.DataFrame,
    owners_brokers_accounts: list[tuple],
) -> pd.DataFrame:

    CURRENT_YEAR = _dt.date.today().year
    TARGET_YEARS = {CURRENT_YEAR, CURRENT_YEAR - 1}

    # --- 年度・配当条件 ---
    base_df = merged_df[
        merged_df["年間配当"].notna()
        & merged_df["年間配当_年度"].notna()
        & merged_df["年間配当_年度"].astype("Int64").isin(TARGET_YEARS)
    ].copy()

    # --- 組条件を OR で作る ---
    cond = False
    for owner, broker, account in owners_brokers_accounts:
        c = True
        if owner not in (None, "*"):
            c = c & (base_df["所有者"] == owner)
        if broker not in (None, "*"):
            c = c & (base_df["証券会社"] == broker)
        if account not in (None, "*"):
            c = c & (base_df["口座区分"] == account)
        cond = cond | c

    result_df = base_df[cond].copy()

    return result_df

# 対象口座の指定
targets = [
    ("Y", "SBI", "NISA"),
    ("Y", "SBI", "特定"),
    ("T", "SBI", "NISA"),
    ("T", "SBI", "特定"),
]

target_df = filter_target_stocks(merged_df, targets)

print("=== 条件一致・集計対象銘柄 ===")
print("件数:", len(target_df))
display(target_df)

,code,所有者,証券会社,口座区分,株数,取得単価,name,industry_33,年間配当,年間配当_年度,年間配当_区分
0,1343,Y,SBI,NISA,90.0000,1779.0,ＮＥＸＴ ＦＵＮＤＳ 東証ＲＥＩＴ指数連動型上場投信,J-REIT,87.6,2025,実績
1,1360,Y,SBI,特定,5000.0000,184.0,ダブルインバース日経,日経ETF,NaN,NaN,NaN
2,1375,Y,SBI,特定,100.0000,1023.0,ユキグニファクトリー,水産・農林業,16.0,2026,予想
3,1488,T,SBI,特定,500.0000,2055.0,ｉＦｒｅｅＥＴＦ 東証ＲＥＩＴ指数,J-REIT,85.0,2025,実績
4,1488,Y,SBI,NISA,104.0000,1731.0,ｉＦｒｅｅＥＴＦ 東証ＲＥＩＴ指数,J-REIT,85.0,2025,実績
5,2003,T,SBI,NISA,100.0000,7018.0,日東富士製粉,食料品,280.0,2026,予想
6,2296,Y,SBI,NISA,100.0000,5627.0,伊藤ハム米久 HD,食料品,320.0,2026,予想
7,2391,Y,SBI,特定,100.0000,1268.0,プラネット,サービス業,44.0,2026,予想
8,2493,Y,SBI,特定,100.0000,925.0,イーサポートリンク,サービス業,5.0,2025,予想
9,2556,Y,SBI,NISA,100.0000,1717.0,Ｏｎｅ ＥＴＦ 東証ＲＥＩＴ指数,J-REIT,84.4,2025,実績


配当取得できない銘柄数: 4


,code,name,industry_33
1,1360,ダブルインバース日経,日経ETF
10,2558,ＭＸＳ米株ＳＰ５００,米国ETF
12,3926,オープンドア,情報・通信業
36,ｅＭＡＸＩＳ Ｓｌｉｍ 米国株式（Ｓ＆Ｐ500）,ｅＭＡＸＩＳ Ｓｌｉｍ 米国株式（Ｓ＆Ｐ500）,米国ETF


=== 条件一致・集計対象銘柄 ===
件数: 33


,code,所有者,証券会社,口座区分,株数,取得単価,name,industry_33,年間配当,年間配当_年度,年間配当_区分
0,1343,Y,SBI,NISA,90.0,1779.0,ＮＥＸＴ ＦＵＮＤＳ 東証ＲＥＩＴ指数連動型上場投信,J-REIT,87.6,2025,実績
2,1375,Y,SBI,特定,100.0,1023.0,ユキグニファクトリー,水産・農林業,16.0,2026,予想
3,1488,T,SBI,特定,500.0,2055.0,ｉＦｒｅｅＥＴＦ 東証ＲＥＩＴ指数,J-REIT,85.0,2025,実績
4,1488,Y,SBI,NISA,104.0,1731.0,ｉＦｒｅｅＥＴＦ 東証ＲＥＩＴ指数,J-REIT,85.0,2025,実績
5,2003,T,SBI,NISA,100.0,7018.0,日東富士製粉,食料品,280.0,2026,予想
6,2296,Y,SBI,NISA,100.0,5627.0,伊藤ハム米久 HD,食料品,320.0,2026,予想
7,2391,Y,SBI,特定,100.0,1268.0,プラネット,サービス業,44.0,2026,予想
8,2493,Y,SBI,特定,100.0,925.0,イーサポートリンク,サービス業,5.0,2025,予想
9,2556,Y,SBI,NISA,100.0,1717.0,Ｏｎｅ ＥＴＦ 東証ＲＥＩＴ指数,J-REIT,84.4,2025,実績
11,2670,T,SBI,特定,100.0,2513.0,エービーシー・マート,小売業,70.0,2026,予想


### (追加1)銘柄名が取得できていない銘柄を抽出

In [2]:
bad = merged_df[
    merged_df["name"].isna() | (merged_df["name"].astype("string") == "0") |
    merged_df["industry_33"].isna() | (merged_df["industry_33"].astype("string").isin(["-", "0"]))
][["code","name","industry_33"]].drop_duplicates()

display(bad)

# IDmap側に存在するか確認
chk = bad.merge(idmap_df[["code","name","industry_33"]], on="code", how="left", suffixes=("_merged","_idmap"))
display(chk)


,code,name,industry_33


,name_merged,industry_33_merged,code,name_idmap,industry_33_idmap


#### IDmapに登録されていない銘柄情報を登録

In [3]:
import os
import pandas as pd
import tempfile

# ============================================================
# #(1) 設定
# ============================================================
IDMAP_CSV = os.path.join("Data", "01_IDmap.csv")

# ============================================================
# #(2) 共通：code 正規化（文字列のまま）
#   - 前後空白除去
#   - 全角数字→半角
# ============================================================
def normalize_code_str(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).strip()
    trans = str.maketrans("０１２３４５６７８９", "0123456789")
    return s.translate(trans)

# ============================================================
# #(3) IDmap 読み込み（BOM対策込み）
# ============================================================
def load_idmap() -> pd.DataFrame:
    df = pd.read_csv(IDMAP_CSV, dtype={"code": "string"}, encoding="utf-8-sig")

    # まれに列名が "﻿code" になる救済
    if "code" not in df.columns:
        for c in df.columns:
            if str(c).replace("\ufeff", "") == "code":
                df = df.rename(columns={c: "code"})
                break

    df["code"] = df["code"].astype("string").map(normalize_code_str)
    return df

# ============================================================
# #(4) IDmap 書き込み（原子置換）
# ============================================================
def save_idmap(df: pd.DataFrame) -> None:
    tmp_dir = os.path.dirname(IDMAP_CSV) or "."
    with tempfile.NamedTemporaryFile(
        "w", delete=False, dir=tmp_dir, suffix=".csv", encoding="utf-8-sig", newline=""
    ) as tf:
        tmp_path = tf.name
        df.to_csv(tf, index=False)
    os.replace(tmp_path, IDMAP_CSV)

# ============================================================
# #(5) 業種一覧表示（表記ゆれ防止）
#   - IDmap上の industry_33 のユニーク一覧を表示
#   - '-' / '0' / 空欄 / NaN は除外
# ============================================================
def show_industry_33_choices() -> list[str]:
    df = load_idmap()

    if "industry_33" not in df.columns:
        print("industry_33 列が IDmap に存在しません。")
        return []

    s = df["industry_33"].astype("string").map(lambda v: v.strip() if isinstance(v, str) else v)
    choices = sorted({v for v in s.dropna().unique().tolist() if v not in ["", "-", "0"]})

    print("=== industry_33 既存候補（表記ゆれ防止用）===")
    for v in choices:
        print("-", v)
    return choices

# ============================================================
# #(6) IDmap を更新/追記する汎用関数
#   - update_idmap(code, name, industry_33) で実行
#   - code 行があれば更新、無ければ新規追加
#   - 他の列は保持
# ============================================================
def update_idmap(code: str, name: str, industry_33: str, *, show_choices: bool = True) -> None:
    code_fixed = normalize_code_str(code)
    name = ("" if name is None else str(name).strip())
    industry_33 = ("" if industry_33 is None else str(industry_33).strip())

    if code_fixed == "":
        raise ValueError("code が空です。")

    df = load_idmap()

    # 必須列が無い場合の保険
    if "name" not in df.columns:
        df["name"] = pd.NA
    if "industry_33" not in df.columns:
        df["industry_33"] = pd.NA

    # 表記ゆれ防止：既存候補を表示＆チェック
    choices = []
    if show_choices:
        choices = show_industry_33_choices()

    if industry_33 and choices and (industry_33 not in choices):
        print(f"\n[注意] industry_33='{industry_33}' は既存候補にありません。新規文字列として保存します。")

    mask = df["code"].astype("string") == code_fixed

    if mask.any():
        df.loc[mask, "name"] = name
        df.loc[mask, "industry_33"] = industry_33
        action = "更新"
    else:
        # 既存列を保ったまま新規行を作る
        new_row = {col: pd.NA for col in df.columns}
        new_row["code"] = code_fixed
        new_row["name"] = name
        new_row["industry_33"] = industry_33
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        action = "追加"

    save_idmap(df)
    print(f"\n[{action}] code={code_fixed} / name='{name}' / industry_33='{industry_33}' を 01_IDmap.csv に反映しました。")

# ============================================================
# #(7) 使い方
# ============================================================
print("=== 使い方 ===")
print("業種一覧を確認：show_industry_33_choices()")
print("追記/更新：update_idmap(code, name, industry_33)")
print("（例）update_idmap('1343', '（銘柄名）', '（industry_33）')")

show_industry_33_choices()
print("")

=== 使い方 ===
業種一覧を確認：show_industry_33_choices()
追記/更新：update_idmap(code, name, industry_33)
（例）update_idmap('1343', '（銘柄名）', '（industry_33）')
=== industry_33 既存候補（表記ゆれ防止用）===
- J-REIT
- その他製品
- その他金融業
- ガラス・土石製品
- ゴム製品
- サービス業
- パルプ・紙
- 不動産業
- 保険業
- 倉庫・運輸関連業
- 化学
- 医薬品
- 卸売業
- 小売業
- 建設業
- 情報・通信業
- 日経ETF
- 機械
- 水産・農林業
- 海運業
- 石油・石炭製品
- 空運業
- 米国ETF
- 精密機器
- 繊維製品
- 証券、商品先物取引業
- 輸送用機器
- 金属製品
- 鉄鋼
- 鉱業
- 銀行業
- 陸運業
- 電気・ガス業
- 電気機器
- 非鉄金属
- 食料品



### (追加2)配当金が記載されていない銘柄の追加　※無配銘柄はそのまま

In [4]:
import os
import re
import tempfile
import pandas as pd

HAITOU_CSV = os.path.join("Data", "03_haitou.csv")

# ============================================================
# #(1) 共通：code 正規化（文字列のまま）
#   - 前後空白除去
#   - 全角数字→半角
# ============================================================
def _normalize_code(x) -> str:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return ""
    s = str(x).strip()
    return s.translate(str.maketrans("０１２３４５６７８９", "0123456789"))

# ============================================================
# #(2) 03_haitou 読み込み（BOM/列名"﻿code"救済込み）
# ============================================================
def _load_haitou() -> pd.DataFrame:
    df = pd.read_csv(HAITOU_CSV, dtype={"code": "string"}, encoding="utf-8-sig")

    # まれに列名が "﻿code" になる救済
    if "code" not in df.columns:
        for c in df.columns:
            if str(c).replace("\ufeff", "") == "code":
                df = df.rename(columns={c: "code"})
                break

    df["code"] = df["code"].astype("string").map(_normalize_code)
    return df

# ============================================================
# #(3) 03_haitou 書き込み（原子置換）
#   - 他データ破損防止のため、必ず一時ファイル経由で置換
# ============================================================
def _save_haitou(df: pd.DataFrame) -> None:
    tmp_dir = os.path.dirname(HAITOU_CSV) or "."
    with tempfile.NamedTemporaryFile(
        "w", delete=False, dir=tmp_dir, suffix=".csv", encoding="utf-8-sig", newline=""
    ) as tf:
        tmp_path = tf.name
        df.to_csv(tf, index=False)
    os.replace(tmp_path, HAITOU_CSV)

# ============================================================
# #(4) 手動入力関数：03に配当を入力/更新
#   引数：code, name, 年度, 配当金額
#   - code行があればその年度セルだけ更新（他は触らない）
#   - code行が無ければ新規行を追加（他列は空のまま）
#   - 年度列が無ければ追加（既存列はそのまま保持）
# ============================================================
def update_haitou_manual(code, name, 年度, 配当金額) -> None:
    code_fixed = _normalize_code(code)
    if code_fixed == "":
        raise ValueError("code が空です。")

    year_str = str(年度).strip()
    if not re.fullmatch(r"\d{4}", year_str):
        raise ValueError(f"年度は '2026' のような4桁で指定してください: 年度={年度!r}")

    # 入力値は「そのまま」保存（数値/文字列どちらもOK）
    # ※ '予 40' のように入れたい場合は 配当金額 にその文字列を渡してください
    if 配当金額 is None or (isinstance(配当金額, float) and pd.isna(配当金額)):
        value = ""  # None/NaNなら空欄にする（消す動作）
    else:
        value = str(配当金額).strip()

    name_str = "" if name is None else str(name).strip()

    df = _load_haitou()

    # name列が無い場合の保険
    if "name" not in df.columns:
        df["name"] = pd.NA

    # 年度列が無ければ追加（他列は一切触らない）
    if year_str not in df.columns:
        df[year_str] = pd.NA
        # 年度列は見やすくするため、(code,name)の後に年度を昇順で並べ替える（非年度列は末尾維持）
        year_cols = sorted([c for c in df.columns if re.fullmatch(r"\d{4}", str(c))], key=lambda x: int(x))
        fixed_cols = [c for c in ["code", "name"] if c in df.columns]
        other_cols = [c for c in df.columns if c not in fixed_cols and c not in year_cols]
        df = df[fixed_cols + year_cols + other_cols]

    mask = df["code"].astype("string") == code_fixed

    if mask.any():
        # 既存行：指定年度セルだけ更新
        df.loc[mask, year_str] = value

        # name は「空欄/NaN/0」のときだけ補完（既存の正しい銘柄名を壊さない）
        cur_name = df.loc[mask, "name"].astype("string")
        need_fill = cur_name.isna() | cur_name.str.strip().isin(["", "0", "-"])
        if name_str and need_fill.any():
            df.loc[mask & need_fill, "name"] = name_str

        action = "更新"
    else:
        # 新規行：既存の列構造を保ったまま追加
        new_row = {col: pd.NA for col in df.columns}
        new_row["code"] = code_fixed
        new_row["name"] = name_str if name_str else pd.NA
        new_row[year_str] = value
        df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
        action = "追加"

    _save_haitou(df)
    print(f"[{action}] 03_haitou.csv に反映: code={code_fixed}, name='{name_str}', 年度={year_str}, 配当金額='{value}'")

# ============================================================
# #(5) 使い方例
# ============================================================
print("使い方: update_haitou_manual(code, name, 年度, 配当金額)")
print("例: update_haitou_manual('3926', 'オープンドア', '2026', '予 10')")
print("例: update_haitou_manual('3926', 'オープンドア', '2025', 8.0)")
print("例: update_haitou_manual('3926', 'オープンドア', '2026', None)  # 空欄に戻す")

使い方: update_haitou_manual(code, name, 年度, 配当金額)
例: update_haitou_manual('3926', 'オープンドア', '2026', '予 10')
例: update_haitou_manual('3926', 'オープンドア', '2025', 8.0)
例: update_haitou_manual('3926', 'オープンドア', '2026', None)  # 空欄に戻す


In [5]:
update_haitou_manual('2556', 'Ｏｎｅ　ＥＴＦ　東証ＲＥＩＴ指数', '2025', 84.4)

[更新] 03_haitou.csv に反映: code=2556, name='Ｏｎｅ　ＥＴＦ　東証ＲＥＩＴ指数', 年度=2025, 配当金額='84.4'
